# Compare NMSLIB Brute Force vs NAPP on the Test Set

This notebook reuses the already trained BM25+Model1 fusion model, exports sparse vectors, runs two test-only NMSLIB passes with the same model (`brute_force` and `napp`), and compares their TREC metrics on the test split.

The notebook assumes the final model has already been trained by the training notebook/script.

## 1) Setup

This notebook starts two NMSLIB servers locally by default: one in `brute_force` mode and one in `napp` mode. You can adjust host/ports in the server cell.

In [ ]:
import csv
import json
import os
import re
import subprocess
from pathlib import Path

REPO_ROOT = Path('/tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP')
SCRIPTS_DIR = REPO_ROOT / 'demo' / 'flexneuart_scripts'
COLLECT_ROOT = (REPO_ROOT / 'demo' / 'collections').resolve()
COLLECT_NAME = 'stackoverflow_all'

TRAIN_PART = 'train'
TEST_PART = 'test'
TRAIN_CAND_QTY = 1000
NMSLIB_CAND_PROV_QTY = 1000

BM25_TUNE_TOP_K = '250'
MODEL1_TUNE_TOP_K = '15'
SELECT_METRIC_COL = 'MAP'

# Local query-server endpoints for two modes
NMSLIB_SERVER_HOST = '127.0.0.1'
NMSLIB_BRUTEFORCE_PORT = 8000
NMSLIB_NAPP_PORT = 8001
BRUTEFORCE_SERVER_URL = f'{NMSLIB_SERVER_HOST}:{NMSLIB_BRUTEFORCE_PORT}'
NAPP_SERVER_URL = f'{NMSLIB_SERVER_HOST}:{NMSLIB_NAPP_PORT}'

BM25_TUNE_TSV = SCRIPTS_DIR / 'bm25tune_stackoverflow_all.tsv'
MODEL1_TUNE_TSV = SCRIPTS_DIR / 'model1tune_stackoverflow_all.tsv'

collect_dir = COLLECT_ROOT / COLLECT_NAME
extractor_rel = Path('exper_desc.best') / 'extractors' / 'bm25_model1_optimal.json'
bruteforce_desc_rel = Path('exper_desc.best') / 'bm25_model1_optimal_bruteforce.json'
napp_desc_rel = Path('exper_desc.best') / 'bm25_model1_optimal_napp.json'
nmslib_rel_dir = Path('nmslib') / 'bm25_model1_interleaved'
docs_export_rel = nmslib_rel_dir / 'docs_export.data'
queries_export_rel = nmslib_rel_dir / f'queries_{TEST_PART}_export.data'

os.environ['COLLECT_ROOT'] = str(COLLECT_ROOT)

def subprocess_env():
    env = os.environ.copy()
    env['COLLECT_ROOT'] = str(COLLECT_ROOT)
    env['PYTHONPATH'] = str(REPO_ROOT) + (':' + env['PYTHONPATH'] if env.get('PYTHONPATH') else '')
    return env

def run_cmd(cmd, cwd=SCRIPTS_DIR):
    print('Running:', ' '.join(str(x) for x in cmd))
    result = subprocess.run(cmd, cwd=cwd, env=subprocess_env(), text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f'Command failed with code {result.returncode}')
    return result

print('REPO_ROOT             =', REPO_ROOT)
print('SCRIPTS_DIR           =', SCRIPTS_DIR)
print('COLLECT_ROOT          =', COLLECT_ROOT)
print('COLLECT_NAME          =', COLLECT_NAME)
print('TRAIN_PART            =', TRAIN_PART)
print('TEST_PART             =', TEST_PART)
print('TRAIN_CAND_QTY        =', TRAIN_CAND_QTY)
print('BRUTEFORCE_SERVER_URL =', BRUTEFORCE_SERVER_URL)
print('NAPP_SERVER_URL       =', NAPP_SERVER_URL)
print('BM25_TUNE_TSV         =', BM25_TUNE_TSV)
print('MODEL1_TUNE_TSV       =', MODEL1_TUNE_TSV)

In [ ]:
def load_best_row(tsv_path: Path, metric_col: str, top_k: str):
    with tsv_path.open('r', encoding='utf-8') as f:
        rows = list(csv.DictReader(f, delimiter='	'))

    if not rows:
        raise RuntimeError(f'No rows in TSV: {tsv_path}')

    filtered = [row for row in rows if row.get('top_k') == str(top_k)]
    if not filtered:
        raise RuntimeError(f'No rows with top_k={top_k} in {tsv_path}')

    best = max(filtered, key=lambda row: float(row[metric_col]))
    return best

def parse_bm25_from_exper_subdir(exper_subdir: str):
    match = re.search(r'k1=([0-9.]+)_b=([0-9.]+)', exper_subdir)
    if not match:
        raise RuntimeError(f'Cannot parse BM25 params from exper_subdir: {exper_subdir}')
    return float(match.group(1)), float(match.group(2))

def parse_model1_from_exper_subdir(exper_subdir: str):
    m_lambda = re.search(r'lambda=([0-9.]+)', exper_subdir)
    m_prob = re.search(r'probSelfTran=([0-9.]+)', exper_subdir)
    m_min = re.search(r'minTranProb=([0-9.eE+-]+)', exper_subdir)

    if not m_lambda or not m_prob:
        raise RuntimeError(f'Cannot parse Model1 params from exper_subdir: {exper_subdir}')

    return float(m_lambda.group(1)), float(m_prob.group(1)), float(m_min.group(1)) if m_min else 2.5e-3

def locate_trained_model():
    expected = collect_dir / 'results' / TEST_PART / 'feat_exper' / 'bm25_model1_optimal' / 'letor' / f'out_{COLLECT_NAME}_{TRAIN_PART}_{TRAIN_CAND_QTY}.model'
    if expected.exists():
        return expected
    matches = sorted((collect_dir / 'results').rglob(f'out_{COLLECT_NAME}_{TRAIN_PART}_{TRAIN_CAND_QTY}.model'))
    if not matches:
        raise FileNotFoundError(f'Could not find trained model file under {collect_dir / "results"}')
    return matches[0]

def parse_rep_file(rep_file: Path):
    metrics = {}
    with rep_file.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or ':' not in line:
                continue
            key, value = line.split(':', 1)
            value = value.strip().split()[0]
            try:
                metrics[key.strip()] = float(value)
            except ValueError:
                continue
    return metrics

def collect_rep_dir(rep_dir: Path):
    rows = []
    for rep_file in sorted(rep_dir.glob('out_*.rep'), key=lambda p: int(p.stem.split('_')[1])):
        top_k = int(rep_file.stem.split('_')[1])
        row = {'top_k': top_k}
        row.update(parse_rep_file(rep_file))
        rows.append(row)
    if not rows:
        raise FileNotFoundError(f'No rep files found in {rep_dir}')
    return rows

bm25_best = load_best_row(BM25_TUNE_TSV, SELECT_METRIC_COL, BM25_TUNE_TOP_K)
model1_best = load_best_row(MODEL1_TUNE_TSV, SELECT_METRIC_COL, MODEL1_TUNE_TOP_K)

BM25_K1, BM25_B = parse_bm25_from_exper_subdir(bm25_best['exper_subdir'])
MODEL1_LAMBDA, MODEL1_PROB_SELF_TRAN, MODEL1_MIN_PROB = parse_model1_from_exper_subdir(model1_best['exper_subdir'])
MODEL_FILE = locate_trained_model()
MODEL_FILE_REL = MODEL_FILE.relative_to(COLLECT_ROOT)
EXTRACTOR_JSON_ABS = collect_dir / extractor_rel
EXACT_DESC_ABS = collect_dir / exact_desc_rel
ANN_DESC_ABS = collect_dir / ann_desc_rel

print('Best BM25 row:')
print(bm25_best)
print('')
print('Best Model1 row:')
print(model1_best)
print('')
print('Selected params:')
print('  BM25_K1 =', BM25_K1)
print('  BM25_B =', BM25_B)
print('  MODEL1_LAMBDA =', MODEL1_LAMBDA)
print('  MODEL1_PROB_SELF_TRAN =', MODEL1_PROB_SELF_TRAN)
print('  MODEL1_MIN_PROB =', MODEL1_MIN_PROB)
print('')
print('Trained model file =', MODEL_FILE)
print('Extractor JSON     =', EXTRACTOR_JSON_ABS)

In [ ]:
extractor_json = [
    {
        'type': 'Model1Similarity',
        'params': {
            'queryFieldName': 'text',
            'indexFieldName': 'text',
            'gizaIterQty': '5',
            'probSelfTran': MODEL1_PROB_SELF_TRAN,
            'lambda': MODEL1_LAMBDA,
            'minModel1Prob': MODEL1_MIN_PROB
        }
    },
    {
        'type': 'TFIDFSimilarity',
        'params': {
            'queryFieldName': 'text',
            'indexFieldName': 'text',
            'similType': 'bm25',
            'k1': BM25_K1,
            'b': BM25_B
        }
    }
]

bruteforce_desc = [
    {
        'experSubdir': 'feat_exper/bm25_model1_optimal_bruteforce',
        'extrTypeFinal': str(extractor_rel).replace('\\', '/'),
        'model_final': str(MODEL_FILE_REL).replace('\\', '/'),
        'cand_prov': 'nmslib',
        'cand_prov_uri': BRUTEFORCE_SERVER_URL,
        'cand_prov_add_conf': json.dumps({
            'extrType': str(extractor_rel).replace('\\', '/'),
            'sparseInterleave': True
        }),
        'cand_prov_qty': NMSLIB_CAND_PROV_QTY,
        'testOnly': 1
    }
]

napp_desc = [
    {
        'experSubdir': 'feat_exper/bm25_model1_optimal_napp',
        'extrTypeFinal': str(extractor_rel).replace('\\', '/'),
        'model_final': str(MODEL_FILE_REL).replace('\\', '/'),
        'cand_prov': 'nmslib',
        'cand_prov_uri': NAPP_SERVER_URL,
        'cand_prov_add_conf': json.dumps({
            'extrType': str(extractor_rel).replace('\\', '/'),
            'sparseInterleave': True
        }),
        'cand_prov_qty': NMSLIB_CAND_PROV_QTY,
        'testOnly': 1
    }
]

BRUTEFORCE_DESC_ABS = collect_dir / bruteforce_desc_rel
NAPP_DESC_ABS = collect_dir / napp_desc_rel

for path, payload in [
    (EXTRACTOR_JSON_ABS, extractor_json),
    (BRUTEFORCE_DESC_ABS, bruteforce_desc),
    (NAPP_DESC_ABS, napp_desc),
]:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2)

print('Wrote extractor JSON:', EXTRACTOR_JSON_ABS)
print('Wrote brute-force descriptor:', BRUTEFORCE_DESC_ABS)
print('Wrote NAPP descriptor:', NAPP_DESC_ABS)

required_inputs = [
    collect_dir / 'input_data' / TRAIN_PART / 'QuestionFields.jsonl',
    collect_dir / 'input_data' / TRAIN_PART / 'AnswerFields.jsonl',
    collect_dir / 'input_data' / TEST_PART / 'QuestionFields.jsonl',
    collect_dir / 'input_data' / TEST_PART / 'AnswerFields.jsonl',
    collect_dir / 'input_data' / TEST_PART / 'qrels.txt',
    collect_dir / 'forward_index' / 'text',
    collect_dir / 'derived_data' / 'giza' / 'text' / 'output.t1.5.bin',
    MODEL_FILE
]

for p in required_inputs:
    if not p.exists():
        raise FileNotFoundError(f'Missing prerequisite: {p}')

print('All prerequisites found.')

In [ ]:
cmd_docs = [
    'bash', './export_nmslib/export_nmslib_sparse.sh',
    COLLECT_NAME,
    str(extractor_rel).replace('\\', '/'),
    str(docs_export_rel).replace('\\', '/'),
    '-model_file',
    str(MODEL_FILE)
]

cmd_queries = [
    'bash', './export_nmslib/export_nmslib_sparse.sh',
    COLLECT_NAME,
    str(extractor_rel).replace('\\', '/'),
    str(queries_export_rel).replace('\\', '/'),
    '-query_file_pref',
    f'{TEST_PART}/QuestionFields'
]

run_cmd(cmd_docs)
run_cmd(cmd_queries)

docs_export_abs = collect_dir / 'derived_data' / docs_export_rel
queries_export_abs = collect_dir / 'derived_data' / queries_export_rel

if not docs_export_abs.exists():
    raise FileNotFoundError(f'Docs export missing: {docs_export_abs}')
if not queries_export_abs.exists():
    raise FileNotFoundError(f'Queries export missing: {queries_export_abs}')

print('Created docs export   =', docs_export_abs)
print('Created queries export=', queries_export_abs)
print('Docs export size      =', docs_export_abs.stat().st_size)
print('Queries export size   =', queries_export_abs.stat().st_size)

## Start NMSLIB servers (brute_force and napp)

This starts two external NMSLIB query servers in background processes: one `brute_force` baseline and one `napp` approximate index.

Update `NMSLIB_QUERY_SERVER_BIN` if the binary lives somewhere else on your machine.

In [ ]:
import socket
import time

NMSLIB_QUERY_SERVER_BIN = Path('query_server')
NMSLIB_SERVER_SIMILARITY = 'negdotprod_sparse_bin_fast'
NMSLIB_SERVER_INDEX_FILE = collect_dir / 'derived_data' / docs_export_rel

if not NMSLIB_QUERY_SERVER_BIN.exists():
    raise FileNotFoundError(
        f'NMSLIB query server binary not found: {NMSLIB_QUERY_SERVER_BIN}. '
        'Set NMSLIB_QUERY_SERVER_BIN to the built query_server executable before running this cell.'
    )

def start_nmslib_server(mode: str, port: int):
    url = f'{NMSLIB_SERVER_HOST}:{port}'
    log_file = collect_dir / 'results' / TEST_PART / f'nmslib_server_{mode}.log'
    pid_file = collect_dir / 'results' / TEST_PART / f'nmslib_server_{mode}.pid'

    server_cmd = [
        str(NMSLIB_QUERY_SERVER_BIN),
        '-p', str(port),
        '-s', NMSLIB_SERVER_SIMILARITY,
        '-i', str(NMSLIB_SERVER_INDEX_FILE),
        '-m', mode,
    ]

    log_file.parent.mkdir(parents=True, exist_ok=True)
    if pid_file.exists():
        existing_pid = pid_file.read_text(encoding='utf-8').strip()
        if existing_pid:
            try:
                os.kill(int(existing_pid), 0)
                print(f'NMSLIB {mode} server already running with PID {existing_pid} on {url}')
                return url, log_file, pid_file
            except OSError:
                pid_file.unlink(missing_ok=True)

    log_handle = log_file.open('a', encoding='utf-8')
    process = subprocess.Popen(
        server_cmd,
        cwd=REPO_ROOT / 'demo',
        stdout=log_handle,
        stderr=log_handle,
        env=subprocess_env(),
    )
    pid_file.write_text(str(process.pid), encoding='utf-8')
    print(f'Started NMSLIB {mode} server:')
    print('  PID =', process.pid)
    print('  URL =', url)
    print('  LOG =', log_file)

    for _ in range(60):
        try:
            with socket.create_connection((NMSLIB_SERVER_HOST, port), timeout=1):
                print(f'NMSLIB {mode} server is accepting connections.')
                break
        except OSError:
            time.sleep(1)
    else:
        raise RuntimeError(f'NMSLIB {mode} server did not start within timeout. See {log_file}')

    return url, log_file, pid_file

BRUTEFORCE_SERVER_URL, BRUTEFORCE_SERVER_LOG, BRUTEFORCE_SERVER_PID = start_nmslib_server('brute_force', NMSLIB_BRUTEFORCE_PORT)
NAPP_SERVER_URL, NAPP_SERVER_LOG, NAPP_SERVER_PID = start_nmslib_server('napp', NMSLIB_NAPP_PORT)

print('BRUTEFORCE_SERVER_URL =', BRUTEFORCE_SERVER_URL)
print('NAPP_SERVER_URL       =', NAPP_SERVER_URL)
print('NMSLIB_SERVER_INDEX_FILE =', NMSLIB_SERVER_INDEX_FILE)
print('NMSLIB_SERVER_SIMILARITY =', NMSLIB_SERVER_SIMILARITY)

In [ ]:
print('Running NMSLIB brute-force test-only evaluation...')
cmd_bruteforce = [
    'bash', './exper/run_experiments.sh',
    COLLECT_NAME,
    str(bruteforce_desc_rel).replace('\\', '/'),
    '-test_part', TEST_PART,
    '-train_part', TRAIN_PART
]
run_cmd(cmd_bruteforce)

bruteforce_rep_dir = collect_dir / 'results' / TEST_PART / 'feat_exper' / 'bm25_model1_optimal_bruteforce' / 'rep'
if not bruteforce_rep_dir.exists():
    raise FileNotFoundError(f'Brute-force report directory not found: {bruteforce_rep_dir}')
print('Brute-force report directory =', bruteforce_rep_dir)

In [ ]:
if not NAPP_SERVER_URL:
    raise RuntimeError('NAPP_SERVER_URL is not set.')

print('Running NMSLIB NAPP test-only evaluation...')
cmd_napp = [
    'bash', './exper/run_experiments.sh',
    COLLECT_NAME,
    str(napp_desc_rel).replace('\\', '/'),
    '-test_part', TEST_PART,
    '-train_part', TRAIN_PART
]
run_cmd(cmd_napp)

napp_rep_dir = collect_dir / 'results' / TEST_PART / 'feat_exper' / 'bm25_model1_optimal_napp' / 'rep'
if not napp_rep_dir.exists():
    raise FileNotFoundError(f'NAPP report directory not found: {napp_rep_dir}')
print('NAPP report directory =', napp_rep_dir)

In [ ]:
def to_rows(rep_dir: Path, label: str):
    rows = collect_rep_dir(rep_dir)
    for row in rows:
        row['system'] = label
    return rows

bruteforce_rows = to_rows(bruteforce_rep_dir, 'brute_force')
napp_rows = to_rows(napp_rep_dir, 'napp') if 'napp_rep_dir' in globals() and napp_rep_dir.exists() else []

metrics_to_show = ['MAP', 'NDCG@10', 'NDCG@20', 'NDCG@100', 'P@20', 'MRR', 'RECALL']
all_top_ks = sorted({row['top_k'] for row in bruteforce_rows} | {row['top_k'] for row in napp_rows})

def metric_value(rows, top_k, metric):
    for row in rows:
        if row['top_k'] == top_k:
            return row.get(metric, float('nan'))
    return float('nan')

header = ['top_k'] + [f'{metric}_bruteforce' for metric in metrics_to_show] + [f'{metric}_napp' for metric in metrics_to_show]
print('\t'.join(header))
for top_k in all_top_ks:
    values = [str(top_k)]
    for metric in metrics_to_show:
        values.append(f'{metric_value(bruteforce_rows, top_k, metric):.6f}')
    for metric in metrics_to_show:
        values.append(f'{metric_value(napp_rows, top_k, metric):.6f}' if napp_rows else 'N/A')
    print('\t'.join(values))

def best_map(rows):
    if not rows:
        return None
    return max(rows, key=lambda row: row.get('MAP', float('-inf')))

best_bruteforce = best_map(bruteforce_rows)
best_napp = best_map(napp_rows) if napp_rows else None
print('')
print('Best brute-force MAP row:', best_bruteforce)
print('Best NAPP MAP row       :', best_napp)

## Notes

- This notebook uses the already trained model and does not retrain.
- Both runs use the same exported sparse vectors and same trained model.
- The only change between the two runs is NMSLIB server mode: `brute_force` vs `napp`.
- The comparison table is based on `.rep` files for each `N`, so both modes are compared at the same cutoff values.